# Объяснение логики проекта

Этот ноутбук нужен как разбор идеи программы, а не просто как запуск кода.

Что делает программа:
- берёт маленькие изображения двух классов: `+` и `V`;
- переводит каждую картинку в набор чисел `0` и `1`;
- подбирает веса простого персептрона;
- подбирает их не обычным обучением, а генетическим алгоритмом;
- после обучения проверяет, насколько хорошо модель различает изображения.


## Общая идея

Здесь решается очень простая задача классификации изображений.

Есть два класса:
- `+` имеет метку `1`;
- `V` имеет метку `0`.

Каждое изображение переводится в чёрно-белый вектор. После этого персептрон считает сумму:

`score = w1*x1 + w2*x2 + ... + wn*xn + bias`

Если сумма больше нуля, модель говорит, что это `+`. Иначе считает, что это `V`.

Главная проблема: нужно найти хорошие веса `w` и смещение `bias`. Этим и занимается генетический алгоритм.


## Зачем нужен датасет

Датасет это набор примеров, на которых модель учится.

В этом проекте датасеты лежат в папках:
- `pictures`
- `more`
- `mix`

Внутри каждой папки есть подпапки `+` и `V`.

Зачем он нужен:
- без датасета модель не знает, какие изображения правильные;
- датасет позволяет считать качество решений;
- по датасету генетический алгоритм понимает, какие наборы весов лучше, а какие хуже.

Именно поэтому функция приспособленности использует `X` и `y`:
- `X` это входные векторы картинок;
- `y` это правильные ответы для этих картинок.


In [ ]:
import os
from PIL import Image

from GA2 import GeneticAlgorithm, GeneticAlgorithmConfig

DATASETS = ("pictures", "more", "mix")
LABELS = {"+": 1, "V": 0}


## Как картинка превращается в числа

Сначала изображение открывается и переводится в оттенки серого. Затем каждый пиксель сравнивается с порогом `128`:
- если пиксель светлый, записывается `1.0`;
- если тёмный, записывается `0.0`.

Так картинка превращается в обычный список чисел. Для алгоритма это уже не изображение, а входной вектор признаков.


In [ ]:
def load_image(path):
    image = Image.open(path).convert("L")
    return [1.0 if pixel >= 128 else 0.0 for pixel in image.tobytes()]


def load_dataset(folder):
    X = []
    y = []
    for class_name, label in LABELS.items():
        class_folder = os.path.join(folder, class_name)
        for filename in sorted(os.listdir(class_folder)):
            if filename.endswith(".png"):
                X.append(load_image(os.path.join(class_folder, filename)))
                y.append(label)
    return X, y


train_X, train_y = load_dataset("more")
print("Количество изображений:", len(train_X))
print("Размер одного вектора:", len(train_X[0]))
print("Первые 25 значений первого изображения:")
print(train_X[0][:25])
print("Метка первого изображения:", train_y[0])


## Логика персептрона

Персептрон в этом проекте очень простой.

Что у него есть:
- список весов `weights`;
- одно смещение `bias`.

Что он делает:
- умножает каждый вход `x[i]` на соответствующий вес `weights[i]`;
- складывает всё это;
- добавляет `bias`;
- принимает решение по знаку результата.

Если результат больше нуля, класс считается равным `1`, иначе `0`.


In [ ]:
def predict(weights, bias, x):
    score = bias
    for i in range(len(x)):
        score += weights[i] * x[i]
    return 1 if score > 0 else 0


def fitness(genome, X, y):
    weights = genome[:-1]
    bias = genome[-1]
    total = 0.0
    for x, target in zip(X, y):
        total += 1.0 if predict(weights, bias, x) == target else -1.0
    return total


## Как здесь работает генетический алгоритм

Обычное обучение персептрона изменяет веса по формуле. Здесь сделано иначе.

Генетический алгоритм работает так:
1. создаёт случайную популяцию хромосом;
2. каждая хромосома хранит все веса и `bias`;
3. для каждой хромосомы считается `fitness`;
4. лучшие решения чаще попадают в дальнейшую работу;
5. из них создаются новые решения через скрещивание и мутацию;
6. после редукции остаётся лучшая часть популяции;
7. через несколько поколений находится удачный набор параметров.

Смысл `fitness` в текущем коде очень простой:
- за правильный ответ даётся `+1`;
- за неправильный ответ даётся `-1`.

Чем выше сумма, тем лучше хромосома распознаёт изображения.


In [ ]:
def train(X, y, generations):
    config = GeneticAlgorithmConfig(
        genome_length=len(X[0]) + 1,
        population_size=60,
        generations=generations,
        elite_size=6,
        tournament_size=4,
        mutation_rate=0.15,
        mutation_strength=0.4,
        crossover_rate=0.7,
        min_value=-3.0,
        max_value=3.0,
        random_seed=42,
    )

    algorithm = GeneticAlgorithm(
        config=config,
        fitness_function=lambda genome: fitness(genome, X, y),
    )

    def show_progress(trace):
        if trace.generation == 0:
            print("Первая популяция:")
            for i, genome in enumerate(trace.population, start=1):
                print(f"{i}: {[round(value, 3) for value in genome]}")
            print("Первая фитнесс функция:")
            print([round(value, 3) for value in trace.fitness_before])
            print()
            return True

        print(f"Поколение {trace.generation}")
        print("Фитнесс до редукции:")
        print([round(value, 3) for value in trace.fitness_before])
        print("Фитнесс после редукции:")
        print([round(value, 3) for value in trace.fitness_after])
        print("Лучшая особь после редукции:")
        print([round(value, 3) for value in trace.best_genome])
        print()
        return True

    best_genome, best_fitness, _ = algorithm.run(progress_callback=show_progress)
    return best_genome[:-1], best_genome[-1], best_fitness


weights, bias, best_fitness = train(train_X, train_y, generations=5)
print("Лучшая приспособленность:", best_fitness)


## Как понимать промежуточный вывод

В программе выводятся несколько важных вещей.

`Первая популяция`:
- это стартовые случайные решения;
- каждый список чисел это один кандидат на хорошие веса.

`Первая фитнесс функция`:
- показывает качество всей стартовой популяции;
- чем больше число, тем лучше особь.

`Фитнесс до редукции`:
- это оценки перед отбором после появления новых особей.

`Фитнесс после редукции`:
- это оценки тех особей, которые остались после отбора.

`Лучшая особь после редукции`:
- это текущий лучший набор весов и `bias`.

То есть по сути ты смотришь, как популяция постепенно становится лучше.


In [ ]:
def accuracy(weights, bias, X, y):
    correct = 0
    for x, target in zip(X, y):
        if predict(weights, bias, x) == target:
            correct += 1
    return correct / len(X)


for name in DATASETS:
    X, y = load_dataset(name)
    print(f"{name}: {accuracy(weights, bias, X, y) * 100:.1f}%")


## Итог

Идея проекта такая:
- изображения переводятся в числа;
- персептрон принимает решение по взвешенной сумме;
- генетический алгоритм ищет хорошие веса;
- качество проверяется по датасету.

То есть это учебный пример того, как можно решить задачу распознавания образов без сложной нейросети и без градиентного обучения.

Если нужно, я могу следующим сообщением ещё сделать вторую версию ноутбука:
- более короткую, как для сдачи;
- или более подробную, как для защиты лабораторной.
